# Rotational interaction

Mode that creates ghost atoms following interactions and guiding interacted atoms to match rotation of the controller:

<video controls src="./assets/rotational_interaction.webm">

## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.mdanalysis import frame_data_to_mdanalysis
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_bundle_path("../systems/openmm/trypsin_benzamidine.openmm.zip")
simulation.load()
universe = frame_data_to_mdanalysis(simulation.make_topology_frame())

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: rotational interaction")
imd_runner.load(0)

In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)
utilities.use_transform_handles()

In [3]:
structure_atoms = universe.select_atoms("not resname BEN")
molecule_atoms = universe.select_atoms("resname BEN")

utilities.selections.update_selection("root", renderer="cartoon")
utilities.selections.update_selection("ligand", renderer="liquorice", particle_ids=molecule_atoms.atoms.indices)

## Interaction hooks

In [4]:
from dataclasses import dataclass

from nanover.imd import ParticleInteraction
from nanover.jupyter import Mode
from nanover.jupyter.ghosts import GhostMoleculeObject
from nanover.jupyter.ghost_follower import GhostFollowerAgent
from nanover.jupyter.transform_grabbing import TransformGrabbingContext


def get_cursor_id_from_interaction(interaction: ParticleInteraction):
    owner_id = interaction.properties.get("owner.id", "")
    hand = interaction.properties.get("label", "hand.").removeprefix("hand.")
    return f"cursor.{owner_id}.{hand}"


@dataclass(kw_only=True)
class GrabData:
    ghost: GhostMoleculeObject
    follower: GhostFollowerAgent


grabbing = TransformGrabbingContext[GrabData].from_utilities(utilities)


class RotationalInteractMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        # try to associate interaction with cursor
        cursor_id = get_cursor_id_from_interaction(interaction)
        cursor = utilities.get_shared_state_value(cursor_id)

        # if there's no associated cursor, ignore
        if cursor is None:
            return

        # make a ghost of the interacted atoms using current particle positions
        ghost = utilities.make_ghost_from_frame_data(
            cursor_id,
            frame_data=imd_runner.app_server.frame_publisher.current_frame,
            atom_indices=interaction.particles,
        )

        # grab the ghost transform with the cursor
        grab = grabbing.start_grab_from_cursor(cursor_id, transform_id=ghost.key, cursor=cursor, scale=False)
        assert grab is not None

        # start a ghost follower agent for this interaction
        follower = GhostFollowerAgent.from_runner(imd_runner)
        follower.setup(key=cursor_id, ghost=ghost)
        follower.start()

        # attach extra data to the grab for later cleanup
        grab.data = GrabData(ghost=ghost, follower=follower)

    def on_interaction_stopped(self, *, key: str, interaction: ParticleInteraction):
        # try to associate interaction with cursor
        cursor_id = get_cursor_id_from_interaction(interaction)
        grab = grabbing.end_grab(cursor_id)

        # if the associated cursor was grabbing, do grab cleanup (stop follower, clear ghost visuals)
        if grab is not None and grab.data is not None:
            grab.data.ghost.clear()
            grab.data.follower.close()

    def on_cursor_updated(self, *, key: str, cursor: dict):
        grabbing.update_grab_from_cursor(key, cursor=cursor)


utilities.modes.add_mode(RotationalInteractMode(), "rotational interaction", icon="🔃")